### Class Board

A escolha da equipa foi criar uma classe que por si só armazene a grid que representa o jogo e também o player atual, por padrão, o jogador que sempre inicia será o jogador 'X'.

O primeiro método a ser introduzido *switch_player* altera a peça a ser jogada no turno, simulando a passagem de turno entre os jogadores.

In [16]:
class Board:
    def __init__(self, rows=6, cols=7):
        self.rows = rows
        self.cols = cols
        self.grid = [[' ' for _ in range(cols)] for _ in range(rows)]
        self.current_player = 'X'
    
    def switch_player(self):
        self.current_player = 'O' if self.current_player == 'X' else 'X'

O próximo método *get_state* foi desenvolvido pensando no registro de estados do tabuleiro para o atendimento da regra de empate e para os algoritmos de Inteligência Artificial.

O método *copy*, que cria uma cópia profunda também é essencial para o desenvolvimento do algoritmo MCTS.

In [17]:
def get_state(self):
    return tuple(tuple(row) for row in self.grid)

def copy(self):
    new_board = Board(self.rows, self.cols)
    new_board.grid = [row[:] for row in self.grid]
    new_board.current_player = self.current_player
    return new_board

# Monkey Patching #
Board.get_state = get_state
Board.copy = copy

Como parte da ideia de movimentação, foi criado o método *get_legal_moves*, responsável por criar e retornar uma lista com todos os movimentos possíveis de drop ou pop. A lista não retorna o estado do tabuleiro se o movimento tivesse sido feito, mas sim um **tuplo** no formato: ("tipo de movimento", coluna)

Tal método também utiliza de outros dois métodos auxiliares: *is_valid_drop* e *is_valid_pop*, que retornam **TRUE** se uma coluna ainda pode receber mais uma peça, e se a peça na linha mais abaixo corresponde a peça do jogador da vez, respectivamente.

In [18]:

def is_valid_drop(self, col):
    return self.grid[0][col] == ' '

def is_valid_pop(self, col, piece):
    return self.grid[self.rows - 1][col] == piece


def get_legal_moves(self):
    legal_moves = []
    for col in range(self.cols):
        if self.is_valid_drop(col):
            legal_moves.append(("push", col))
        
        if self.is_valid_pop(col, self.current_player):
            legal_moves.append(("pop", col))
    
    return legal_moves

# Monkey Patching #
Board.is_valid_drop = is_valid_drop
Board.is_valid_pop = is_valid_pop
Board.get_legal_moves = get_legal_moves

Ainda sobre a movimentação, com a lista de movimentos possíveis implementada, constriu-se o método *apply_move*, o qual lê o tuplo representativo do movimento escolhido e utiliza de **três** outros métodos auxiliares.

Drop: utiliza-se o método *get_next_open_row*, que identifica a linha que deverá ser inserida a peça numa determinada coluna, e *drop_piece*, que insere a peça no local desejado.

Pop: o método *pop_piece* itera sobre a coluna a que está sendo feito o movimento descendo todos os seus elementos em 1 nível(linha) e insere um espaço vazio na linha 0 daquela coluna.

Ao fim, o método *apply_move* passa a jogada para o outro jogador.

In [19]:

def get_next_open_row(self, col):
    for r in range(self.rows - 1, -1, -1):
        if self.grid[r][col] == ' ':
            return r
    return None

def drop_piece(self, col, piece):
    row = self.get_next_open_row(col)
    if row is not None:
        self.grid[row][col] = piece
    return row

def pop_piece(self, col):
    for r in range(self.rows - 1, 0, -1):
        self.grid[r][col] = self.grid[r - 1][col]
    self.grid[0][col] = ' '

def apply_move(self, move):
    move_type, col = move

    if move_type == "push":
        self.drop_piece(col, self.current_player)
    elif move_type == "pop":
        self.pop_piece(col)
        
    self.switch_player()

# Monkey Patching #

Board.get_next_open_row = get_next_open_row
Board.drop_piece = drop_piece
Board.pop_piece = pop_piece
Board.apply_move = apply_move

Os últimos dois métodos criados são relacionados a checagem de vitória e a checagem de eventual empate.

*check_win*: recebe uma peça ("X" ou "O") e verifica nas 4 direções se houve um alinhamento quádruplo dessa peça, retorna **TRUE** se houve.

*is_full*: verifica o topo de cada coluna, se todos forem diferente de ' ' retorna **TRUE**.

In [20]:
def check_win(self, piece):
    """Verifica todas as combinações de vitória para a peça dada."""
    # Horizontal
    for c in range(self.cols - 3):
        for r in range(self.rows):
            if (self.grid[r][c] == piece and self.grid[r][c+1] == piece and 
                self.grid[r][c+2] == piece and self.grid[r][c+3] == piece):
                return True
    # Vertical
    for c in range(self.cols):
        for r in range(self.rows - 3):
            if (self.grid[r][c] == piece and self.grid[r+1][c] == piece and 
                self.grid[r+2][c] == piece and self.grid[r+3][c] == piece):
                return True
    # Diagonal Positiva
    for c in range(self.cols - 3):
        for r in range(self.rows - 3):
            if (self.grid[r][c] == piece and self.grid[r+1][c+1] == piece and 
                self.grid[r+2][c+2] == piece and self.grid[r+3][c+3] == piece):
                return True
    # Diagonal Negativa
    for c in range(self.cols - 3):
        for r in range(3, self.rows):
            if (self.grid[r][c] == piece and self.grid[r-1][c+1] == piece and 
                self.grid[r-2][c+2] == piece and self.grid[r-3][c+3] == piece):
                return True
    return False

def is_full(self):
    """Verifica se o tabuleiro está completamente cheio."""
    for c in range(self.cols):
        if self.grid[0][c] == ' ':
            return False
    return True

# Monkey Patching #

Board.check_win = check_win
Board.is_full = is_full